# 02 · Exploratory Data Analysis — OULAD

**DSP391m · Group 1 · FPT University** · *Time-Aware Explainable ML for Early At-Risk Student Detection*.

**Self-contained EDA:** the chart style, statistical helpers and every analysis function are defined in this notebook (no `src/` imports). We use `master_raw` (t=100%) for univariate/bivariate/correlation, and the six checkpoint datasets (built in notebook 03, loaded from `data/checkpoints/`) for the time-aware analysis.

## Executive summary

On the master table (**32,593** records, at-risk rate **52.8%**):

1. **Behaviour dominates demographics** — engagement & assessment separate the classes strongly (Cohen's *d* up to **2.55**; 19/19 numeric features significant at *q*<0.05); demographic association is weak (Cramér's *V* ≤ **0.15**).
2. **Clickstream is strongly right-skewed** (skew up to ~35) → motivates `log1p`.
3. **`days_since_last_activity` is bimodal** — the signature of disengagement.
4. **No leakage** — no feature |r| ≥ 0.95 with the label; two engagement pairs are multicollinear (|r| ≥ 0.8), flagged for SHAP.
5. **Signal is early** — `n_days_active` and `days_since_last_activity` reach a large effect (*d* ≥ 0.8) by **10%** of course length, scores by **20%** (RQ1).

## 0. Objectives & statistical method

**Method:** Mann–Whitney U (non-parametric, fits the skew) + Benjamini–Hochberg correction & **Cohen's d** for numeric features; **chi-square + Cramér's V** for categorical; **Pearson/Spearman** for multivariate structure. Charts follow the team standard (300 dpi, colour-blind safe).

In [ ]:
import os, json, logging, warnings
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('nb')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
INTERIM_DIR = ROOT / 'data' / 'interim'
CHECKPOINTS_DIR = ROOT / 'data' / 'checkpoints'
CHECKPOINT_MAP_PATH = ROOT / 'data' / 'checkpoint_map.csv'
REPORTS_DIR = ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'
FIGURES_DIR = REPORTS_DIR / 'figures'
CHECKPOINTS = (10, 20, 40, 60, 80, 100)
RANDOM_SEED = 42
for _d in (INTERIM_DIR, CHECKPOINTS_DIR, TABLES_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator
from IPython.display import Image, display
from scipy import stats

### Chart standard (Task 39)

In [ ]:
CLASS_COLOURS = {0: "#2166AC", 1: "#C0392B"}  # 0 not-at-risk (blue), 1 at-risk (red)


CLASS_LABELS = {0: "Not-at-risk", 1: "At-risk"}


GROUP_COLOURS = {
    "Demographic": "#7F8C8D",  # muted grey
    "Engagement": "#C0392B",  # red (behavioural)
    "Performance": "#2166AC",  # blue (assessment)
}


_RC = {
    # resolution
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    # typography
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlepad": 8,
    "axes.labelsize": 10.5,
    "axes.labelcolor": "#222222",
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "xtick.color": "#444444",
    "ytick.color": "#444444",
    "legend.fontsize": 9.5,
    "legend.frameon": False,
    "figure.titlesize": 14,
    "figure.titleweight": "bold",
    # axes / spines / grid (less chart-junk)
    "axes.edgecolor": "#888888",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#E6E6E6",
    "grid.linewidth": 0.7,
}


def apply_style() -> None:
    """Apply the team chart standard to the current matplotlib session."""
    sns.set_theme(style="whitegrid", context="notebook")
    plt.rcParams.update(_RC)


_THOUSANDS = FuncFormatter(lambda x, _pos: f"{x:,.0f}")


def tidy_axis(ax: plt.Axes, *, nbins: int = 5, integer: bool = False) -> None:
    """De-clutter a numeric x-axis: few round ticks + thousands separators + despine.

    Prevents the overlapping tick labels that plague wide-range count features
    (e.g. ``total_clicks`` spanning 0–24,000).
    """
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins, integer=integer))
    ax.xaxis.set_major_formatter(_THOUSANDS)
    sns.despine(ax=ax)


def savefig(fig: plt.Figure, name: str) -> Path:
    """Save a figure to reports/figures/<name>.png at print resolution (300 dpi)."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    return path

### Feature groups & statistical helpers

In [ ]:
DEMOGRAPHIC_NUM = ["num_of_prev_attempts", "studied_credits", "date_registration"]


ENGAGEMENT_NUM = [
    "total_clicks",
    "n_days_active",
    "max_clicks_single_day",
    "mean_clicks_per_active_day",
    "days_since_last_activity",
    "clicks_forumng",
    "clicks_oucontent",
    "clicks_resource",
    "clicks_homepage",
    "clicks_oucollaborate",
    "clicks_quiz",
    "clicks_subpage",
    "clicks_url",
]


PERFORMANCE_NUM = [
    "mean_score_to_date",
    "weighted_score_to_date",
    "n_assessments_submitted",
]


NUMERIC_ALL = DEMOGRAPHIC_NUM + ENGAGEMENT_NUM + PERFORMANCE_NUM


NUMERIC_MAIN = [
    "total_clicks",
    "n_days_active",
    "mean_clicks_per_active_day",
    "max_clicks_single_day",
    "days_since_last_activity",
    "mean_score_to_date",
    "weighted_score_to_date",
    "n_assessments_submitted",
    "num_of_prev_attempts",
    "studied_credits",
]


CATEGORICAL_MAIN = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]


GROUP_OF = {
    **{c: "Demographic" for c in DEMOGRAPHIC_NUM},
    **{c: "Engagement" for c in ENGAGEMENT_NUM},
    **{c: "Performance" for c in PERFORMANCE_NUM},
}


GROUP_COLOUR = {
    "Demographic": "#7F8C8D",
    "Engagement": "#C0392B",
    "Performance": "#2166AC",
}

In [ ]:
def load_master(path: Path = INTERIM_DIR / "master_raw.parquet") -> pd.DataFrame:
    return pd.read_parquet(path)


def load_checkpoints(checkpoints_dir: Path = CHECKPOINTS_DIR) -> pd.DataFrame | None:
    frames = []
    for t in CHECKPOINTS:
        p = checkpoints_dir / f"dataset_t{t}.parquet"
        if not p.exists():
            return None
        frames.append(pd.read_parquet(p))
    return pd.concat(frames, ignore_index=True)


def _save_table(df: pd.DataFrame, name: str, index: bool = True) -> None:
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(TABLES_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")


def _cohens_d(a: pd.Series, b: pd.Series) -> float:
    """Pooled-SD standardised mean difference |Cohen's d|."""
    va, vb = a.var(ddof=1), b.var(ddof=1)
    pooled = np.sqrt((va + vb) / 2) or 1.0
    return float(abs(a.mean() - b.mean()) / pooled)


def _benjamini_hochberg(pvals: list[float]) -> list[float]:
    """BH-adjusted p-values (q-values) controlling the false discovery rate."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(ranked, 0, 1)
    return out.tolist()


def _cramers_v(confusion: np.ndarray) -> float:
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.sum()
    r, k = confusion.shape
    return float(np.sqrt(chi2 / (n * (min(r, k) - 1)))) if min(r, k) > 1 else 0.0

### Analysis functions

Each function writes its table to `reports/tables/` and its figure to `reports/figures/`, and returns a small findings dict. Defining them here keeps the logic visible and editable, while the per-section cells below run and narrate them.

In [ ]:
def data_quality(master: pd.DataFrame) -> dict:
    apply_style()
    miss = master.isnull().sum()
    profile = pd.DataFrame(
        {
            "dtype": master.dtypes.astype(str),
            "n_missing": miss,
            "pct_missing": (miss / len(master) * 100).round(2),
            "n_unique": master.nunique(),
        }
    ).sort_values("pct_missing", ascending=False)
    _save_table(profile, "data_quality_profile")

    miss_cols = profile[profile["n_missing"] > 0]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(
        miss_cols.index[::-1], miss_cols["pct_missing"][::-1], color=CLASS_COLOURS[1]
    )
    ax.set_xlabel("% missing")
    ax.set_title("Missing values by column (only columns with gaps)")
    savefig(fig, "quality_missingness")
    plt.close(fig)
    return {
        "n_records": int(len(master)),
        "n_columns": int(master.shape[1]),
        "columns_with_missing": miss_cols["n_missing"].to_dict(),
    }


def univariate(master: pd.DataFrame) -> dict:
    apply_style()
    desc = master[NUMERIC_ALL].describe().T
    desc["median"] = master[NUMERIC_ALL].median()
    desc["skew"] = master[NUMERIC_ALL].skew()
    desc["kurtosis"] = master[NUMERIC_ALL].kurtosis()
    desc = desc.round(3)
    _save_table(desc, "univariate_numeric")

    for col in CATEGORICAL_MAIN:
        freq = (
            master[col]
            .value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="count")
        )
        freq["pct"] = (freq["count"] / len(master) * 100).round(2)
        _save_table(freq, f"freq_{col}", index=False)

    # Histograms + KDE for the main numeric features, coloured by group.
    fig, axes = plt.subplots(2, 5, figsize=(20, 8.6), constrained_layout=True)
    for ax, col in zip(axes.ravel(), NUMERIC_MAIN):
        sns.histplot(
            master[col],
            kde=True,
            ax=ax,
            color=GROUP_COLOUR[GROUP_OF[col]],
            edgecolor="white",
            linewidth=0.3,
            alpha=0.9,
        )
        ax.set_title(col, fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.text(
            0.95,
            0.93,
            f"skew {master[col].skew():.1f}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8.5,
            color="#555555",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#CCCCCC", lw=0.6),
        )
        tidy_axis(ax, nbins=4)
    fig.suptitle(
        "Univariate distributions of numeric features (histogram + KDE)\n"
        "colour = feature group — Engagement (red) · Performance (blue) · Demographic (grey)",
        fontweight="bold",
    )
    savefig(fig, "univariate_hist_kde")
    plt.close(fig)

    fig, axes = plt.subplots(2, 5, figsize=(20, 8.6), constrained_layout=True)
    for ax, col in zip(axes.ravel(), NUMERIC_MAIN):
        sns.boxplot(y=master[col], ax=ax, color=GROUP_COLOUR[GROUP_OF[col]])
        ax.set_title(col, fontsize=10)
        ax.set_ylabel("")
    fig.suptitle("Univariate boxplots (IQR outlier inspection)", fontweight="bold")
    savefig(fig, "univariate_boxplots")
    plt.close(fig)

    # Categorical frequency grid.
    fig, axes = plt.subplots(2, 3, figsize=(18, 10.5), constrained_layout=True)
    for ax, col in zip(axes.ravel(), CATEGORICAL_MAIN):
        vc = master[col].value_counts().sort_values()
        ax.barh(vc.index.astype(str), vc.values, color="#4D4D4D")
        ax.set_title(f"{col} ({master[col].nunique()} levels)")
        ax.tick_params(axis="y", labelsize=8)
        sns.despine(ax=ax)
    fig.suptitle("Categorical frequency distributions", fontweight="bold")
    savefig(fig, "univariate_categorical_freq")
    plt.close(fig)

    return {
        "most_skewed": desc["skew"]
        .abs()
        .sort_values(ascending=False)
        .head(5)
        .round(2)
        .to_dict(),
        "most_leptokurtic": desc["kurtosis"]
        .sort_values(ascending=False)
        .head(5)
        .round(2)
        .to_dict(),
    }


def target_distribution(master: pd.DataFrame) -> dict:
    apply_style()
    counts = master["final_result"].value_counts()
    rate = float(master["at_risk"].mean())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    order = ["Distinction", "Pass", "Fail", "Withdrawn"]
    axes[0].bar(
        order,
        [counts.get(k, 0) for k in order],
        color=[CLASS_COLOURS[0], CLASS_COLOURS[0], CLASS_COLOURS[1], CLASS_COLOURS[1]],
    )
    axes[0].set_title("final_result")
    axes[0].set_ylabel("Students")
    axes[0].tick_params(axis="x", rotation=20)
    binary = master["at_risk"].map(CLASS_LABELS).value_counts()
    axes[1].bar(
        binary.index,
        binary.values,
        color=[
            CLASS_COLOURS[1] if "At" in i else CLASS_COLOURS[0] for i in binary.index
        ],
    )
    axes[1].set_title(f"at_risk (positive rate = {rate:.1%})")
    fig.suptitle("Target distribution and class imbalance", fontweight="bold")
    savefig(fig, "target_distribution")
    plt.close(fig)
    return {
        "final_result_counts": counts.to_dict(),
        "at_risk_rate": round(rate, 4),
        "imbalance_ratio": round(rate / (1 - rate), 3),
    }


def numeric_vs_target(master: pd.DataFrame) -> dict:
    apply_style()
    g1 = master[master["at_risk"] == 1]
    g0 = master[master["at_risk"] == 0]

    rows, pvals = [], []
    for col in NUMERIC_ALL:
        a, b = (
            g1[col].dropna(),
            g0[col].dropna(),
        )  # master_raw has a few raw NaNs (e.g. date_registration)
        d = _cohens_d(a, b)
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        rows.append(
            {
                "feature": col,
                "group": GROUP_OF[col],
                "mean_not_at_risk": round(b.mean(), 3),
                "mean_at_risk": round(a.mean(), 3),
                "cohens_d": round(d, 3),
                "mannwhitney_U": float(u),
                "p_value": p,
            }
        )
        pvals.append(p)
    table = pd.DataFrame(rows)
    table["p_adj_bh"] = _benjamini_hochberg(pvals)
    table["significant_(q<0.05)"] = table["p_adj_bh"] < 0.05
    table = table.sort_values("cohens_d", ascending=False).reset_index(drop=True)
    _save_table(table, "bivariate_numeric_tests", index=False)

    # Effect-size figure (ranked |Cohen's d|, coloured by group, values labelled).
    fig, ax = plt.subplots(figsize=(9.5, 7))
    t = table.iloc[::-1]
    bars = ax.barh(
        t["feature"],
        t["cohens_d"],
        color=[GROUP_COLOUR[g] for g in t["group"]],
        edgecolor="white",
        linewidth=0.4,
    )
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8, color="#333333")
    for x, ls in [(0.2, ":"), (0.5, "--"), (0.8, "-")]:
        ax.axvline(x, color="grey", ls=ls, lw=1)
    ax.set_xlim(0, float(t["cohens_d"].max()) * 1.12)
    ax.set_xlabel("|Cohen's d|  (dotted .2 small · dashed .5 medium · solid .8 large)")
    ax.set_title("Discriminative power of numeric features (at-risk vs not-at-risk)")
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOUR[g])
        for g in ("Engagement", "Performance", "Demographic")
    ]
    ax.legend(
        handles,
        ("Engagement", "Performance", "Demographic"),
        loc="lower right",
        title="Feature group",
    )
    sns.despine(ax=ax)
    savefig(fig, "bivariate_effect_sizes")
    plt.close(fig)

    # Boxplots of the six strongest features.
    top = table.head(6)["feature"].tolist()
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    for ax, col in zip(axes.ravel(), top):
        sns.boxplot(
            x=master["at_risk"].map(CLASS_LABELS),
            y=master[col],
            ax=ax,
            hue=master["at_risk"].map(CLASS_LABELS),
            palette={
                CLASS_LABELS[0]: CLASS_COLOURS[0],
                CLASS_LABELS[1]: CLASS_COLOURS[1],
            },
            legend=False,
        )
        ax.set_title(col)
        ax.set_xlabel("")
    fig.suptitle("Six most discriminative numeric features by class", fontweight="bold")
    savefig(fig, "bivariate_top_boxplots")
    plt.close(fig)

    return {
        "top5_by_cohens_d": table.head(5)[["feature", "cohens_d", "p_adj_bh"]].to_dict(
            "records"
        ),
        "n_significant": int(table["significant_(q<0.05)"].sum()),
        "n_tested": int(len(table)),
    }


def categorical_vs_target(master: pd.DataFrame) -> dict:
    apply_style()
    rows = []
    for col in CATEGORICAL_MAIN:
        ct = pd.crosstab(master[col].fillna("Unknown"), master["at_risk"])
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        rows.append(
            {
                "feature": col,
                "chi2": round(chi2, 2),
                "dof": dof,
                "p_value": p,
                "cramers_v": round(_cramers_v(ct.values), 3),
            }
        )
    table = (
        pd.DataFrame(rows)
        .sort_values("cramers_v", ascending=False)
        .reset_index(drop=True)
    )
    _save_table(table, "bivariate_categorical_tests", index=False)

    overall = master["at_risk"].mean()
    fig, axes = plt.subplots(2, 3, figsize=(18, 10.5), constrained_layout=True)
    for ax, col in zip(axes.ravel(), CATEGORICAL_MAIN):
        rate = master.groupby(col)["at_risk"].mean().sort_values()
        v = table.loc[table["feature"] == col, "cramers_v"].iloc[0]
        ax.barh(rate.index.astype(str), rate.values, color=CLASS_COLOURS[1])
        ax.axvline(overall, color="grey", ls="--", lw=1)
        ax.set_title(f"At-risk rate by {col}  (Cramer's V={v})")
        ax.set_xlabel("At-risk rate")
        ax.tick_params(axis="y", labelsize=8)
        sns.despine(ax=ax)
    fig.suptitle(
        "At-risk rate across categorical levels (dashed = overall 52.8%)",
        fontweight="bold",
    )
    savefig(fig, "bivariate_categorical_rate")
    plt.close(fig)
    return {"association_cramers_v": table.set_index("feature")["cramers_v"].to_dict()}


def correlation(master: pd.DataFrame) -> dict:
    apply_style()
    num = master[NUMERIC_MAIN + ["at_risk"]]
    for method, fname in [("pearson", "corr_pearson"), ("spearman", "corr_spearman")]:
        corr = num.corr(method=method)
        fig, ax = plt.subplots(figsize=(10.5, 8.5))
        sns.heatmap(
            corr,
            annot=True,
            fmt=".2f",
            cmap="RdBu_r",
            center=0,
            vmin=-1,
            vmax=1,
            square=True,
            linewidths=0.5,
            linecolor="white",
            annot_kws={"size": 8},
            cbar_kws={"label": f"{method.title()} r", "shrink": 0.82},
            ax=ax,
        )
        ax.set_title(
            f"{method.title()} correlation of numeric features (incl. at_risk)", pad=12
        )
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        plt.setp(ax.get_yticklabels(), rotation=0)
        savefig(fig, fname)
        plt.close(fig)

    pear = num.corr("pearson")
    pairs = (
        pear.where(~np.eye(len(pear), dtype=bool))
        .abs()
        .unstack()
        .dropna()
        .sort_values(ascending=False)
    )
    strong = [
        {"a": a, "b": b, "r": round(pear.loc[a, b], 3)}
        for (a, b), v in pairs.items()
        if a < b and v >= 0.6
    ]
    _save_table(pd.DataFrame(strong), "correlation_strong_pairs", index=False)

    target_corr = (
        pear["at_risk"].drop("at_risk").sort_values(key=np.abs, ascending=False)
    )
    _save_table(
        target_corr.round(3).rename("pearson_r_with_at_risk").to_frame(),
        "correlation_with_target",
    )
    fig, ax = plt.subplots(figsize=(8, 6))
    tc = target_corr.iloc[::-1]
    ax.barh(
        tc.index,
        tc.values,
        color=[CLASS_COLOURS[1] if v > 0 else CLASS_COLOURS[0] for v in tc.values],
    )
    ax.set_xlabel("Pearson r with at_risk")
    ax.set_title("Correlation of numeric features with the target")
    savefig(fig, "corr_with_target")
    plt.close(fig)

    multicollinear = [p for p in strong if abs(p["r"]) >= 0.8]
    leakage = [c for c, v in target_corr.abs().items() if v >= 0.95]
    return {
        "strong_pairs_ge_0.6": strong[:12],
        "multicollinear_ge_0.8": multicollinear,
        "top_target_corr": target_corr.abs().head(8).round(3).to_dict(),
        "leakage_suspects_ge_0.95": leakage,
    }


def time_aware(checkpoints: pd.DataFrame | None) -> dict:
    if checkpoints is None:
        return {"status": "checkpoints_not_found"}
    apply_style()
    feats = [
        "total_clicks",
        "n_days_active",
        "mean_score_to_date",
        "n_assessments_submitted",
    ]

    # (a) Mean trajectory by class.
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)
    for ax, col in zip(axes, feats):
        grp = checkpoints.groupby(["t_percent", "at_risk"])[col].mean().unstack()
        for lab in (0, 1):
            ax.plot(
                grp.index,
                grp[lab],
                marker="o",
                color=CLASS_COLOURS[lab],
                label=CLASS_LABELS[lab],
            )
        ax.set_title(f"Mean {col}")
        ax.set_xlabel("Course progress (%)")
        ax.legend()
    fig.suptitle(
        "Mean feature trajectory by class across checkpoints", fontweight="bold"
    )
    savefig(fig, "time_mean_trajectory")
    plt.close(fig)

    # (b) Discrimination curve: |Cohen's d| per feature at each checkpoint (RQ1).
    disc_rows = []
    for t in CHECKPOINTS:
        sub = checkpoints[checkpoints["t_percent"] == t]
        g1, g0 = sub[sub["at_risk"] == 1], sub[sub["at_risk"] == 0]
        for col in feats + ["days_since_last_activity", "weighted_score_to_date"]:
            disc_rows.append(
                {
                    "t_percent": t,
                    "feature": col,
                    "cohens_d": round(_cohens_d(g1[col], g0[col]), 3),
                }
            )
    disc = pd.DataFrame(disc_rows)
    _save_table(
        disc.pivot(index="t_percent", columns="feature", values="cohens_d"),
        "discrimination_by_checkpoint",
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    for col in disc["feature"].unique():
        s = disc[disc["feature"] == col]
        ax.plot(s["t_percent"], s["cohens_d"], marker="o", label=col)
    ax.axhline(0.8, color="grey", ls="--", lw=1)
    ax.set_xlabel("Course progress (%)")
    ax.set_ylabel("|Cohen's d| (class separability)")
    ax.set_title("How feature discrimination grows across checkpoints (RQ1)")
    ax.legend(fontsize=8, ncol=2)
    savefig(fig, "time_discrimination_curve")
    plt.close(fig)

    earliest = {}
    for col in disc["feature"].unique():
        s = disc[disc["feature"] == col].sort_values("t_percent")
        hit = s[s["cohens_d"] >= 0.8]
        earliest[col] = int(hit["t_percent"].iloc[0]) if len(hit) else None
    return {
        "discrimination_by_checkpoint": disc.pivot(
            index="t_percent", columns="feature", values="cohens_d"
        ).to_dict(),
        "earliest_checkpoint_d_ge_0.8": earliest,
    }


def withdrawn_analysis(master: pd.DataFrame) -> dict:
    apply_style()
    m = master.copy()
    m["status"] = np.where(
        m["final_result"] == "Withdrawn",
        "Withdrawn",
        np.where(m["at_risk"] == 1, "Fail", "Not-at-risk"),
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    sns.boxplot(
        x="status",
        y="days_since_last_activity",
        data=m,
        ax=axes[0],
        order=["Not-at-risk", "Fail", "Withdrawn"],
        color=CLASS_COLOURS[1],
    )
    axes[0].set_title("Inactivity gap by outcome")
    sns.boxplot(
        x="status",
        y="total_clicks",
        data=m,
        ax=axes[1],
        order=["Not-at-risk", "Fail", "Withdrawn"],
        color=CLASS_COLOURS[0],
    )
    axes[1].set_title("Total clicks by outcome")
    fig.suptitle(
        "Withdrawn students: the activity-decay early-warning signal (Option A)",
        fontweight="bold",
    )
    savefig(fig, "withdrawn_activity_decay")
    plt.close(fig)
    return {
        "median_days_since_last_activity": m.groupby("status")[
            "days_since_last_activity"
        ]
        .median()
        .round(1)
        .to_dict(),
        "median_total_clicks": m.groupby("status")["total_clicks"]
        .median()
        .round(1)
        .to_dict(),
    }

### Load the data

In [ ]:
master = load_master()
checkpoints = load_checkpoints()
print('master:', master.shape, '| checkpoints stacked:',
      None if checkpoints is None else checkpoints.shape)

## 1. Dataset profile (structure-first)

For every column: type, completeness, cardinality and (numeric) distribution shape — flags exactly what cleaning must address.

In [ ]:
num_cols = master.select_dtypes("number").columns
profile = pd.DataFrame({
    "dtype": master.dtypes.astype(str),
    "n_missing": master.isnull().sum(),
    "pct_missing": (master.isnull().mean() * 100).round(2),
    "n_unique": master.nunique(),
    "skew": master[num_cols].skew().round(2),
}).sort_values("pct_missing", ascending=False)
print(f'{len(master):,} rows x {master.shape[1]} cols | numeric={len(num_cols)} | '
      f"at-risk rate={master['at_risk'].mean():.1%}")
display(profile)

## 2. Data quality

Only three columns have gaps: `date_unregistration` (structural — students who never withdraw), `imd_band` (~3.4%, filled `Unknown`), `date_registration` (~0.14%, train-median imputed). After cleaning: **0 NaN** in any feature column.

In [ ]:
_ = data_quality(master)
display(pd.read_csv(TABLES_DIR / 'data_quality_profile.csv', index_col=0))
display(Image(filename=str(FIGURES_DIR / 'quality_missingness.png')))

## 3. Univariate analysis

Most clickstream features are **strongly right-skewed** (heavy tails are real behaviour) → `log1p`, not row deletion. `clicks_resource` skew ≈ 34.7; `days_since_last_activity` is **bimodal**. Non-normal ⇒ later tests are non-parametric.

In [ ]:
_ = univariate(master)
num = pd.read_csv(TABLES_DIR / 'univariate_numeric.csv', index_col=0)
display(num[['mean','std','min','50%','max','skew','kurtosis']])
display(Image(filename=str(FIGURES_DIR / 'univariate_hist_kde.png')))

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'univariate_boxplots.png')))

**Categorical.** 83% of students hold A-Level or below; Post-Graduate / No-Formal are rare (<1.1%) → one-hot with `handle_unknown=ignore`.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'univariate_categorical_freq.png')))

## 4. Target distribution & class imbalance

At-risk rate **52.8%** (imbalance ratio 1.12): a *slight majority*, not severe imbalance. PR-AUC and recall on the at-risk class are the headline metrics. **Withdrawn** is the largest single class.

In [ ]:
_ = target_distribution(master)
counts = master['final_result'].value_counts().rename_axis('final_result').reset_index(name='count')
counts['pct'] = (counts['count'] / len(master) * 100).round(1)
counts['label'] = counts['final_result'].isin(['Fail','Withdrawn']).map(
    {True:'at-risk (1)', False:'not-at-risk (0)'})
display(counts)
display(Image(filename=str(FIGURES_DIR / 'target_distribution.png')))

## 5. Bivariate — numeric features vs the target

Cohen's d + Mann–Whitney U (BH-corrected). At n≈32k almost every p is tiny → **rank by effect size**. Strongest: `days_since_last_activity` (*d*=2.55), `n_assessments_submitted` (2.05), `weighted_score_to_date` (1.96). All 19/19 significant.

In [ ]:
_ = numeric_vs_target(master)
tb = pd.read_csv(TABLES_DIR / 'bivariate_numeric_tests.csv').sort_values('cohens_d', ascending=False)
display(tb[['feature','group','mean_not_at_risk','mean_at_risk','cohens_d','p_adj_bh','significant_(q<0.05)']].head(10))
print(f"Significant (q<0.05): {int(tb['significant_(q<0.05)'].sum())}/{len(tb)}")
display(Image(filename=str(FIGURES_DIR / 'bivariate_effect_sizes.png')))

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'bivariate_top_boxplots.png')))

## 6. Bivariate — categorical features vs the target (Cramér's V)

Chi-square is significant (large *n*) but **Cramér's V** is small: `highest_education` (0.15) and `imd_band` (0.15) lead, `gender` (0.02) is negligible. Demographics → keep for **fairness analysis**, not prediction.

In [ ]:
_ = categorical_vs_target(master)
display(pd.read_csv(TABLES_DIR / 'bivariate_categorical_tests.csv'))
display(Image(filename=str(FIGURES_DIR / 'bivariate_categorical_rate.png')))

## 7. Multivariate — correlation, multicollinearity, leakage

Top |r| with the label: `days_since_last_activity` **+0.78** (agreeing with Cohen's d). **No feature |r| ≥ 0.95** ⇒ no leakage; two multicollinear pairs ≥ 0.8 flagged for SHAP.

In [ ]:
_ = correlation(master)
print('Strong pairs (|r| >= 0.6):')
display(pd.read_csv(TABLES_DIR / 'correlation_strong_pairs.csv'))
print('Correlation with the target (top |r|):')
display(pd.read_csv(TABLES_DIR / 'correlation_with_target.csv', index_col=0))
display(Image(filename=str(FIGURES_DIR / 'corr_pearson.png')))

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'corr_with_target.png')))

## 8. Time-aware analysis — when does the signal emerge? (RQ1)

|Cohen's d| per feature at six checkpoints 10%→100%. First to reach *d* ≥ 0.8: `n_days_active` **and** `days_since_last_activity` from **10%**, scores/submissions from **20%** → early prediction is feasible from ~10–20% of course length.

> Requires the six `dataset_t*.parquet` files (notebook 03). If absent, this section is skipped.

In [ ]:
if checkpoints is not None:
    ta = time_aware(checkpoints)
    display(pd.read_csv(TABLES_DIR / 'discrimination_by_checkpoint.csv', index_col=0).round(3))
    print('Earliest checkpoint reaching d>=0.8:', ta['earliest_checkpoint_d_ge_0.8'])
    display(Image(filename=str(FIGURES_DIR / 'time_discrimination_curve.png')))
else:
    print('checkpoints not found — run notebook 03 first')

In [ ]:
if checkpoints is not None:
    display(Image(filename=str(FIGURES_DIR / 'time_mean_trajectory.png')))

## 9. The Withdrawn early-warning signal (Step-0 Option A)

Median days-since-last-activity rises **11 → 116 (Fail) → 233 (Withdrawn)**; median total clicks collapse ~1,425 → ~89. *Caveat:* withdrawn students are trivially separable at **late** checkpoints — the genuine test of the model is at the **early** ones.

In [ ]:
_ = withdrawn_analysis(master)
m = master.copy()
m['status'] = np.where(m['final_result'].eq('Withdrawn'), 'Withdrawn',
                       np.where(m['at_risk'].eq(1), 'Fail', 'Not-at-risk'))
summary = m.groupby('status').agg(
    median_days_idle=('days_since_last_activity', 'median'),
    median_total_clicks=('total_clicks', 'median'),
    n=('at_risk', 'size')).reindex(['Not-at-risk','Fail','Withdrawn'])
display(summary.round(1))
display(Image(filename=str(FIGURES_DIR / 'withdrawn_activity_decay.png')))

## 10. Findings & implications for modelling

1. **Early signal (RQ1)** — behaviour/performance separate from 10–20%; 40–60% is a robust, actionable window.
2. **Behaviour ≫ demographics (RQ1/RQ2)** — engagement/assessment reach *d* > 2; demographics small (V ≤ 0.15) → SHAP should rank behaviour highest.
3. **Mild imbalance (RQ3)** — 52.8% at-risk → RQ3 weighs SMOTE/ADASYN/class-weight on PR-AUC/recall.
4. **Correlated features (RQ2)** — two multicollinear pairs may destabilise explanation importance.
5. **No leakage** — no near-perfect correlate; the time-aware cut removes future events.

**References.** [1] Kuzilek et al., *Scientific Data* 4:170171, 2017. [2] Adnan et al., *IEEE Access* 9:7519–7539, 2021. [3] Tomasevic et al., *Computers & Education* 143:103676, 2020.